# Page 60: pystripe-first preprocessing test

現行の `offset → FFC → pystripe → morphology` に対し、`offset → pystripe → FFC → morphology` を60枚目で比較します。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
import numpy as np
import pystripe
from scipy.ndimage import gaussian_filter1d
import tifffile as tiff

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Project root containing src/ was not found.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from preprocess import flatfield_like_correction, subtract_background_morphology

INPUT_TIF = PROJECT_ROOT / 'data' / '042926_MAY08R_FOS_1_retake_c.tif'
CURRENT_OUTPUT = PROJECT_ROOT / 'outputs' / '042926_MAY08R_FOS_1_retake_c_uint16_scale10000' / '042926_MAY08R_FOS_1_retake_c_060.tif'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'preprocess_order_test'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PYSTRIPE_FIRST_OUTPUT = OUTPUT_DIR / '042926_MAY08R_FOS_1_retake_c_060_pystripe_first.tif'
COMPARISON_PNG = OUTPUT_DIR / '042926_MAY08R_FOS_1_retake_c_060_order_comparison.png'

In [ ]:
with tiff.TiffFile(str(INPUT_TIF)) as tif:
    raw = tif.pages[59].asarray().astype(np.float32)

offset = np.percentile(raw, 0.05)
raw0 = raw - offset
raw0[raw0 < 0] = 0

# Destripe before the spatially varying FFC gain is applied.
destriped = pystripe.filter_streaks(
    raw0, sigma=(128, 256), level=7, wavelet='db2'
)
ffc, field = flatfield_like_correction(
    destriped, sigma=120, reference_level=100, max_gain=3.0
)
preprocessed, background = subtract_background_morphology(ffc, radius=20)

output_uint16 = np.clip(preprocessed * 100.0, 0, 65535).astype(np.uint16)
tiff.imwrite(str(PYSTRIPE_FIRST_OUTPUT), output_uint16)
print('Saved:', PYSTRIPE_FIRST_OUTPUT)

## Horizontal-band metric

1. 同じROI（`y=1000:3500, x=300:1900`）を使う。
2. 各行についてX方向の中央値を取り、行輝度プロファイルを作る。
3. Gaussian（σ=30行）で遅い解剖学的変化を推定して差し引く。
4. 残差の標準偏差をROIのp1–p99輝度幅で割り、百分率にする。

広範囲に連続する横帯は残り、局所的な細胞点は行中央値で抑えられます。解剖構造も一部寄与するため、絶対的なstripe量ではなく同一画像・ROI間の比較指標です。

In [ ]:
def horizontal_band_metric(image):
    roi = np.asarray(image[1000:3500, 300:1900], dtype=np.float32)
    row_profile = np.median(roi, axis=1)
    slow_trend = gaussian_filter1d(row_profile, sigma=30)
    high_frequency_residual = row_profile - slow_trend
    robust_range = np.percentile(roi, 99) - np.percentile(roi, 1)
    return 100.0 * np.std(high_frequency_residual) / robust_range

current = tiff.imread(str(CURRENT_OUTPUT))
pystripe_first = tiff.imread(str(PYSTRIPE_FIRST_OUTPUT))

print('Current order metric       : {:.4f}%'.format(horizontal_band_metric(current)))
print('Pystripe-first order metric: {:.4f}%'.format(horizontal_band_metric(pystripe_first)))

In [ ]:
norm = PowerNorm(gamma=0.7, vmin=0, vmax=5500)
fig, axes = plt.subplots(1, 2, figsize=(12, 12), dpi=160, facecolor='black')
for ax, image, title in zip(
    axes,
    (current, pystripe_first),
    ('Current: FFC → pystripe', 'Test: pystripe → FFC'),
):
    ax.imshow(image, cmap='gray', norm=norm)
    ax.set_title('{}\nstripe metric = {:.4f}%'.format(title, horizontal_band_metric(image)), color='white')
    ax.axis('off')
plt.tight_layout()
fig.savefig(str(COMPARISON_PNG), facecolor='black', bbox_inches='tight')
plt.show()
print('Saved:', COMPARISON_PNG)